In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


# Prepare for Training

In [2]:
!pip install transformers
!pip install wandb
!pip install av
#Model Configuration
!pip install evaluate
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate
!pip install datasets
# prompt: install pytorch
!pip install torch torchvision torchaudio
!pip install python-dotenv

#For Model
from huggingface_hub import hf_hub_download
from transformers import TrainingArguments
from transformers import Trainer, TrainingArguments, AdamW
from transformers import Trainer
import wandb
#For Training
import torch
from transformers import  VivitConfig,VivitForVideoClassification
import evaluate
import os
from dotenv import load_dotenv
load_dotenv()
import av
from datasets import Dataset
#For Data Prep
import numpy as np
import cv2
from pathlib import Path
import pandas as pd
import json



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.0/33.0 MB 57.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 22.8 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transform

In [17]:
sample_size = 24

def read_video_pyav(container, indices):
    '''
    Decode the video with PyAV decoder.
    Args:
        container (`av.container.input.InputContainer`): PyAV container.
        indices (`List[int]`): List of frame indices to decode.
    Returns:
        result (np.ndarray): np array of decoded frames of shape (num_frames, height, width, 1).
    '''
    ##DEBUG start
    #print(f"Debug - indicies are {indices}")

    frames = []
    container.seek(0)
    start_index = indices[0]
    end_index = indices[-1]
    for i, frame in enumerate(container.decode(video=0)):
        if i > end_index:
            break
        if i >= start_index and i in indices:
            #print(f"Debug - Processing frame {i}")
            reformatted_frame = frame.reformat(width=224, height=224)
            frames.append(reformatted_frame)
    new=np.stack([x.to_ndarray(format="rgb24") for x in frames])
    #new = np.stack([np.repeat(x.to_ndarray(format="gray")[..., np.newaxis], 3, axis=-1) for x in frames])
    if len(frames)!=sample_size:
      print(f"Debug - Frames is {len(frames)} & New shape is {new.shape}")
    return new


def sample_frame_indices(total_frames, clip_len):
    '''
    Sample a given number of frame indices from the video.
    Args:
        total_frames (`int`): Total number of frames in the video.
        clip_len (`int`): How many frames to sample from the middle section of the video
    Returns:
        indices (`List[int]`): List of sampled frame indices
    '''
    '''
        if total_frames > 100:
          start_idx = int(total_frames*0.1) #Ignore first 10% of the frames
          end_idx = int(total_frames*0.9) #Ignore last 10% of the frames
        else:
          start_idx = 0
          end_idx = total_frames
    '''

    start_idx = 0
    end_idx = total_frames

    indices = np.linspace(start_idx, end_idx, num=clip_len)
    indices = np.clip(indices, start_idx, end_idx - 1).astype(np.int64)
    #print(f"Debug - Indices size is - {indices.size}. Start index - {start_idx} End index - {end_idx}")
    return indices

def frames_convert_and_create_dataset_dictionary(csv_path,video_folder):
    class_labels = []
    all_videos=[]

    # Read the CSV into a pandas DataFrame
    # $batch_size = Load the first 161 gloss
    #df = pd.read_csv(csv_path, sep='|', skiprows=0, nrows=161)
    df = pd.read_csv(csv_path, sep='|')

    # Create the class_labels list (unique glosses from the CSV)
    class_labels = df['gloss'].unique().tolist()

    # Create the all_videos list of dictionaries
    # Initialize an empty list for all_videos
    all_videos = []

    # Loop through each row in the DataFrame
    for index, row in df.iterrows():
        # Extract the file_name and remove '.mp4' to get the video_id
        video_id = row['video_id']

        # Construct the full path to the video file
        video_path = os.path.join(video_folder, video_id)
        # Try opening with 'latin-1' encoding
        try:
            container = av.open(video_path)
        except UnicodeDecodeError:
            # Handle the case where both encodings fail
            print(f"Error for {video_path}. Skipping this video.")
            continue

        #skip the video if there isn't at least 20 frames
        nframes = list(container.decode(video=0))
        total_frames = len(nframes)
        clip_len = sample_size
        if total_frames > clip_len:
          print(f"{row['gloss']} - Processing file {video_path} number of Frames: {total_frames}")
          indices = sample_frame_indices(total_frames, clip_len)
          video = read_video_pyav(container, indices)

          # Extract the gloss (label)
          gloss = row['gloss']

          # Create a dictionary with the 'video' and 'labels' keys
          video_entry = {
              'video': video,
              'labels': gloss
          }

          # Append the dictionary to the all_videos list
          all_videos.append(video_entry)
        else:
          print(f"{row['gloss']} - Skipping file {video_path} number of Frames: {total_frames}")

    return all_videos, class_labels

# Training
❌ **DO NOT RUN**
**This code will run for 11 hours on a L4 GPU.**

0. Read the csv file - video_gloss_stat
1.

In [20]:
# prompt: check the device type if cuda

if torch.cuda.is_available():
  device = torch.device("cuda")
  print("Using CUDA device:", torch.cuda.get_device_name(0))
else:
  device = torch.device("cpu")
  print("CUDA is not available, using CPU.")


Using CUDA device: NVIDIA L4


In [21]:
#Call the loop

# Path to the output CSV from the previous program
input_csv = "/content/drive/Othercomputers/My MacBook Pro/ASL_Citizen/video_gloss_stat_v1004.csv"  # Replace with your CSV file path
video_folder = '/content/drive/MyDrive/videos_50_trimmed'

# Add error handling and potential solutions
video_dict, class_labels = frames_convert_and_create_dataset_dictionary(input_csv,video_folder)


HELLO - Skipping file /content/drive/MyDrive/videos_50_trimmed/1340650432799635-HELLO_trim.mp4 number of Frames: 11
WATER - Skipping file /content/drive/MyDrive/videos_50_trimmed/40723216723050504-WATER_trim.mp4 number of Frames: 13
NO - Skipping file /content/drive/MyDrive/videos_50_trimmed/09714329486323847-NO_trim.mp4 number of Frames: 14
HELP - Skipping file /content/drive/MyDrive/videos_50_trimmed/8646796456836814-HELP_trim.mp4 number of Frames: 19
SORRY - Skipping file /content/drive/MyDrive/videos_50_trimmed/1636390875008531-SORRY_trim.mp4 number of Frames: 17
WATER - Skipping file /content/drive/MyDrive/videos_50_trimmed/6099837370552477-WATER_trim.mp4 number of Frames: 14
HELLO - Skipping file /content/drive/MyDrive/videos_50_trimmed/6448349493816601-HELLO_trim.mp4 number of Frames: 13
NEED - Skipping file /content/drive/MyDrive/videos_50_trimmed/28071486967073533-NEED_trim.mp4 number of Frames: 16
HELLO - Skipping file /content/drive/MyDrive/videos_50_trimmed/5254351100197965

In [22]:
#Model Configuration

metric = evaluate.load("accuracy", trust_remote_code=True)
def compute_metrics(p):
    return metric.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([(torch.tensor(x['pixel_values']))  for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])}


def initialise_model(shuffled_dataset, device="cpu", model="google/vivit-b-16x2-kinetics400"):
    """initialize model
    """
    labels = shuffled_dataset['train'].features['labels'].names
    config = VivitConfig.from_pretrained(model)
    config.num_classes=len(labels)
    config.id2label = {str(i): c for i, c in enumerate(labels)}
    config.label2id = {c: str(i) for i, c in enumerate(labels)}
    config.num_frames=24
    config.video_size= [24, 224, 224]

    model = VivitForVideoClassification.from_pretrained(
    model,
    ignore_mismatched_sizes=True,
    config=config,).to(device)
    return model

The following pre-processing works fine with smalled datasets. In case you have less class labels, or Gloss in that case.

In [23]:
#Pre-Processing
from transformers import VivitImageProcessor, VivitModel

image_processor = VivitImageProcessor.from_pretrained("google/vivit-b-16x2-kinetics400")

def process_example(example):
    inputs = image_processor(list(np.array(example['video'])), return_tensors='pt')
    inputs['labels'] = example['labels']
    return inputs



def create_dataset(video_dictionary):
    dataset = Dataset.from_list(video_dictionary)
    dataset = dataset.class_encode_column("labels")
    processed_dataset = dataset.map(process_example, batched=False)
    processed_dataset=processed_dataset.remove_columns(['video'])
    shuffled_dataset= processed_dataset.shuffle(seed=42)
    shuffled_dataset= shuffled_dataset.map(lambda x: {'pixel_values': torch.tensor(x['pixel_values']).squeeze()})
    shuffled_dataset =shuffled_dataset.train_test_split(test_size=0.1)
    return shuffled_dataset

The following pre-processing works with larger datasets. In case you have more class labels, or Gloss in that case.

**Don't run this. It fails.**

In [ ]:
#Pre-Processing
from datasets import Dataset, concatenate_datasets
import torch
import numpy as np
from transformers import VivitImageProcessor, VivitModel

# Initialize the Vivit image processor
image_processor = VivitImageProcessor.from_pretrained("google/vivit-b-16x2-kinetics400")

# Define the function to process each example
def process_example(example):
    inputs = image_processor(list(np.array(example['video'])), return_tensors='pt')
    inputs['labels'] = example['labels']
    return inputs

# Function to create dataset from smaller batches
def create_dataset(video_dictionary, batch_size=50):
    # Split the video_dictionary into smaller batches
    sub_datasets = []
    num_batches = len(video_dictionary) // batch_size + (len(video_dictionary) % batch_size > 0)
    print(f"Number of batches: {num_batches}")

    for i in range(num_batches):
        print(f"Processing batch {i + 1}/{num_batches}")
        # Create a smaller batch of the dictionary
        batch_dict = video_dictionary[i * batch_size:(i + 1) * batch_size]
        # Create a dataset from the smaller batch
        batch_dataset = Dataset.from_list(batch_dict)
        # Encode the labels
        batch_dataset = batch_dataset.class_encode_column("labels")
        # Process each example in the batch
        processed_batch = batch_dataset.map(process_example, batched=False)
        # Remove the video column to save memory
        processed_batch = processed_batch.remove_columns(['video'])
        sub_datasets.append(processed_batch)

    # Concatenate all processed sub-datasets
    processed_dataset = concatenate_datasets(sub_datasets)
    # Shuffle and split the dataset
    shuffled_dataset = processed_dataset.shuffle(seed=42)
    shuffled_dataset = shuffled_dataset.map(lambda x: {'pixel_values': torch.tensor(x['pixel_values']).squeeze()})
    shuffled_dataset = shuffled_dataset.train_test_split(test_size=0.1)

    return shuffled_dataset


In [24]:
shuffled_dataset = create_dataset(video_dict)

Casting to class labels:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

Map:   0%|          | 0/105 [00:00<?, ? examples/s]

In [25]:
class_labels = sorted(class_labels)
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"Unique classes: {list(label2id.keys())}.")

Unique classes: ['FAMILY', 'FRIEND', 'HELLO', 'HELP', 'NEED', 'NO', 'PLEASE', 'SORRY', 'WATER', 'YES'].


In [26]:
shuffled_dataset['train']

Dataset({
    features: ['labels', 'pixel_values'],
    num_rows: 94
})

In [27]:
model = initialise_model(shuffled_dataset, device)

Some weights of VivitForVideoClassification were not initialized from the model checkpoint at google/vivit-b-16x2-kinetics400 and are newly initialized because the shapes did not match:
- vivit.embeddings.position_embeddings: found shape torch.Size([1, 3137, 768]) in the checkpoint and torch.Size([1, 2353, 768]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([10]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
from transformers import TrainingArguments, Trainer

training_output_dir = "/content/drive/MyDrive/ASL_Citizen_vocab10_Model"
training_args = TrainingArguments(
        output_dir=training_output_dir,
        num_train_epochs=5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=1,
        learning_rate=1e-05,
        #learning_rate=5e-05,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        seed=42,
        eval_strategy="steps",
        eval_steps=10,
        warmup_steps=int(0.1 * 20),
        optim="adamw_torch",
        lr_scheduler_type="linear",
        fp16=True,
        gradient_accumulation_steps=2,
        report_to="wandb"
    )

#optimizer = torch.optim.AdamW(model.parameters(), lr=5e-05, betas=(0.9, 0.999), eps=1e-08)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-05, betas=(0.9, 0.999), eps=1e-08,weight_decay=0.01)

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=shuffled_dataset["train"],
    eval_dataset=shuffled_dataset["test"],
    optimizers=(optimizer, None),
    compute_metrics = compute_metrics
)

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [29]:
wandb_key =  os.getenv("WANDB_API_KEY")
wandb.login(key=wandb_key)

PROJECT = "ViViT_ASL_Sample"
MODEL_NAME = "google/vivit-b-16x2-kinetics400"
DATASET = "ASL Citizen vocab 10"

wandb.init(project=PROJECT, # the project I am working on
           tags=[MODEL_NAME, DATASET],
           notes ="Fine tuning ViViT with ASL Citizen sample")

In [30]:
torch.cuda.empty_cache()

In [31]:
with wandb.init(project=PROJECT, job_type="train", # the project I am working on
           tags=[MODEL_NAME, DATASET],
           notes =f"Fine tuning {MODEL_NAME} with {DATASET}."):
           train_results = trainer.train()

Traceback (most recent call last):
  File "<ipython-input-31-f82b285ff035>", line 4, in <cell line: 1>
    train_results = trainer.train()
  File "/usr/local/lib/python3.10/dist-packages/transformers/trainer.py", line 2079, in train
    return inner_training_loop(
  File "/usr/local/lib/python3.10/dist-packages/transformers/trainer.py", line 2415, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs)
  File "/usr/local/lib/python3.10/dist-packages/transformers/trainer.py", line 3518, in training_step
    loss = self.compute_loss(model, inputs)
  File "/usr/local/lib/python3.10/dist-packages/transformers/trainer.py", line 3565, in compute_loss
    outputs = model(**inputs)
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 1553, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/torch/nn/modules/module.py", line 1562, in _call_impl
    return forward_call(*args, **kwargs)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 22.17 GiB of which 20.88 MiB is free. Process 4911 has 22.14 GiB memory in use. Of the allocated memory 21.88 GiB is allocated by PyTorch, and 26.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [19]:
trainer.save_model("model")
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)
trainer.save_state()

***** train metrics *****
  epoch                    =       4.918
  total_flos               = 706153397GF
  train_loss               =      2.3369
  train_runtime            =  0:21:58.92
  train_samples_per_second =       0.462
  train_steps_per_second   =       0.114


In [20]:
custom_path = "./model"
with wandb.init(project=PROJECT, job_type="models"):
  torch.save(model.state_dict(), "model.pth1004v3")
  artifact = wandb.Artifact("ViViT-Fine-tuned", type="model")
  artifact.add_dir(custom_path)
  artifact.add_file("model.pth1004v3")
  wandb.save(custom_path)
  wandb.log_artifact(artifact)
  #run.link_artifact(artifact, f"indraneel-raxit-home-org/wandb-registry-model/asl_vivit_sample")

wandb: Adding directory to artifact (./model)... Done. 1.1s


In [21]:
from pathlib import Path
import wandb

run = wandb.init(project="collection-linking-quickstart")

artifact_filepath = Path("./my_model_artifact.txt")
artifact_filepath.write_text("simulated model file")

logged_artifact = run.log_artifact(
  artifact_filepath,
  "artifact-name",
  type="model"
)
run.link_artifact(
  artifact=logged_artifact,
  target_path="indraneel-raxit-home-org/wandb-registry-model/asl_vivit_sample"
)
run.finish()

# Inference

In [42]:
file_name = "/content/drive/MyDrive/ASL_Citizen_10_Test_trimmed/IMG_8912_Please_trim.mp4"
container = av.open(file_name)

In [23]:
import wandb
run = wandb.init()
artifact = run.use_artifact('indraneel-raxit-home/ViViT_ASL_Sample/ViViT-Fine-tuned:v8', type='model')
artifact_dir = artifact.download()

wandb: Downloading large artifact ViViT-Fine-tuned:v8, 667.28MB. 4 files... 
wandb:   4 of 4 files downloaded.  
Done. 0:0:0.5


In [24]:
artifact_dir

'/content/artifacts/ViViT-Fine-tuned:v8'

In [25]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [26]:
from transformers import VivitImageProcessor, VivitForVideoClassification
image_processor = VivitImageProcessor.from_pretrained("google/vivit-b-16x2-kinetics400")
config = VivitConfig.from_pretrained(artifact_dir)
fine_tune_model = VivitForVideoClassification.from_pretrained(artifact_dir,config=config)

In [ ]:
#moviepy.editor.ipython_display(container.name)

In [43]:
indices = sample_frame_indices(total_frames=container.streams.video[0].frames, clip_len=sample_size)
print(f"Processing file {file_name} number of Frames: {container.streams.video[0].frames}")
video = read_video_pyav(container=container, indices=indices)
inputs = image_processor(list(video), return_tensors="pt")

Processing file /content/drive/MyDrive/ASL_Citizen_10_Test_trimmed/IMG_8912_Please_trim.mp4 number of Frames: 79


In [44]:
with torch.no_grad():
    outputs = fine_tune_model(**inputs)
    logits = outputs.logits

In [45]:
predicted_label = logits.argmax(-1).item()
prediction = fine_tune_model.config.id2label[predicted_label]
prediction

'WATER'